In [3]:
import pandas as pd
import numpy as np


In [39]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### 1. String to Lowercase

In [40]:
df["review"] = df['review'].str.lower()
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 2. Removing the URLs

In [41]:
import re

def remove_urls(text):
    return re.sub(r"http\S+", "", text)
df["review"] = df['review'].apply(remove_urls)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 3. Removing HTML Tag

In [43]:
def remove_html_tags(text):
    return re.sub(r"<.*?>", "", text) #remove HTML tags

df['review'] = df['review'].apply(remove_html_tags)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 4. Removing Punctuations

In [44]:
def remove_punctuation(text):
    return re.sub(r"[^A-Za-z0-9\s]", "", text) #^ means exclude; remove other than letters, numbers and whitespace

df['review'] = df['review'].apply(remove_punctuation)

In [45]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. Removing the StopWords

In [46]:
import nltk #natural language toolkit
nltk.download("punkt")# Download the punkt tokenizer
nltk.download("stopwords")# Download the stopwords
nltk.download("punkt_tab")# Download the punkt tokenizer for tab-separated values


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...


[nltk_data]   Package punkt_tab is already up-to-date!


True

In [47]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [48]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    filtered_tokens = [token for token in tokens if token.lower() not in stop_words]
    return " ".join(filtered_tokens)
df['review'] = df['review'].apply(remove_stopwords)

In [49]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


### 6. Stemming
 - running -> run
 - played -> play

In [50]:
from nltk.stem import PorterStemmer

def stemmer(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df['review'] = df['review'].apply(stemmer)

In [51]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


### 7. Encoding 

In [53]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['sentiment'] = le.fit_transform(df['sentiment'])
y=df['sentiment']

### 8. Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df['review'])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4079393 stored elements and shape (50000, 5000)>

### 9. Datasets and DataLoader

In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [64]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [66]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

C:\Users\HP\AppData\Local\Temp\ipykernel_5492\2931448922.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(y_train.values).float()


In [67]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

### 10. Build Recurrent Neural Network

In [70]:
import torch.nn as nn
import torch.optim as optim

In [71]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        #fully connected layer
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        #optional => shape(num_layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])
        return out

In [72]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### 11. Training the RNN

In [73]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch+1}/{epochs} and loss = {loss.item()}")

epoch 1/10 and loss = 0.25058823823928833
epoch 2/10 and loss = 0.3683871626853943
epoch 3/10 and loss = 0.16002477705478668
epoch 4/10 and loss = 0.1478666365146637
epoch 5/10 and loss = 0.15536820888519287
epoch 6/10 and loss = 0.2821996808052063
epoch 7/10 and loss = 0.17867238819599152
epoch 8/10 and loss = 0.18110793828964233
epoch 9/10 and loss = 0.5173962116241455
epoch 10/10 and loss = 0.23400628566741943


### 12. Evaluation

In [75]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        output = model(Xb)
        predicted = (torch.sigmoid(output.squeeze()) > 0.5).float()
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()
    print(f"accurray = {correct_vals / tot_vals * 100}")

accurray = 87.32
